In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS schema.bronze;

In [0]:
catalog = "schema"
source_schema = "azure_blob_storage"
bronze_schema = "bronze"

In [0]:
import re

def pascal_to_snake(name):
    """Convert PascalCase to snake_case"""
    snake = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', name)
    return re.sub('([a-z0-9])([A-Z])', r'\1_\2', snake).lower()

data_dict_df = spark.table(f"{catalog}.{source_schema}.data_dictionary")

column_metadata = data_dict_df.filter(
    (data_dict_df.field.isNotNull()) & 
    (~data_dict_df.field.startswith("_"))
).select("table", "field")

table_names = [row.table for row in column_metadata.select("table").distinct().collect()]

print(f"Found {len(table_names)} tables to ingest: {table_names}")

for table_name in table_names:
    dict_columns = [
        row.field for row in column_metadata.filter(column_metadata.table == table_name).collect()
    ]
    
    dict_columns_snake = [pascal_to_snake(col) for col in dict_columns]
    
    print(f"\nProcessing table: {table_name}")
    print(f"  Columns from dictionary: {dict_columns}")
    print(f"  Converted to snake_case: {dict_columns_snake}")
    
    # Read the source table
    source_table = f"{catalog}.{source_schema}.{table_name}"
    df = spark.table(source_table)
    
    # ignoring the Fivetran columns
    actual_columns = [col for col in df.columns if not col.startswith("_")]
    
    columns_to_select = []
    for dict_col in dict_columns_snake:
        # Find matching column in actual table
        matching = [ac for ac in actual_columns if ac.lower() == dict_col.lower()]
        if matching:
            columns_to_select.append(matching[0])
        else:
            print(f"Warning: Column '{dict_col}' not found in source table")
    
    print(f"Columns to ingest: {columns_to_select}")
    
    df_filtered = df.select(*columns_to_select)
    
    # bronze table path
    bronze_table = f"{catalog}.{bronze_schema}.{table_name}"
    
    df_filtered.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(bronze_table)
    
    row_count = spark.table(bronze_table).count()
    print(f"  ✓ Ingested {row_count} rows to {bronze_table}")

print(f"\nAll tables ingested to {catalog}.{bronze_schema}")